# Wave Field Animation: Dielectric Cylinders

Generates a wave field visualization for a slab of **dielectric cylinders** (n=1.3).

**What this example demonstrates:**
1. Computing the S-matrix for dielectric cylinders
2. Building the wave field in three regions: reflection, slab interior, transmission
3. Normal incidence vs. optimal wavefront (SVD of S11)
4. Plotting the transmission eigenvalue distribution $f(\tau)$

In [ ]:
import sys
import time
import numpy as np
import matplotlib.pyplot as plt
from numpy.linalg import svd
from scipy.special import hankel2 as hankel2_scipy

sys.path.insert(0, '../..')

from Scattering_Code.smatrix_parameters import smatrix_parameters
from Scattering_Code.smatrix import smatrix, _vall_batch, _extract_scalar
from Scattering_Code.transall import transall
from Scattering_Code.sall import sall
from Scattering_Code.ky import ky


In [ ]:
WAVELENGTH = 0.93
PERIOD     = 12.81
RADIUS     = 0.25
N_CYL_REF  = 1.3
MU         = 1.0
CMMAX      = 5
PHIINC     = np.pi / 2
Eva_TOL    = 1e-2
NUM_CYL    = 50
SEED       = 42
GRID_RES   = 200
PR_Y       = 7

n_prop = int(np.floor(PERIOD / WAVELENGTH))
n_eva  = max(int(np.floor(
    PERIOD / (2*np.pi) * np.sqrt(
        (np.log(Eva_TOL) / (2*RADIUS))**2 + (2*np.pi/WAVELENGTH)**2
    )
)) - n_prop, 0)
nmax = n_prop + n_eva
nm   = 2 * nmax + 1

## 1. Setup and Compute S-Matrix

In [ ]:
spacing = 2.5 * RADIUS
cyls_per_row = int(PERIOD / spacing)
rows_needed = NUM_CYL / cyls_per_row + 2
thickness = round(max(0.5, rows_needed * spacing * 1.5), 1)

rng = np.random.RandomState(SEED)
margin = RADIUS * 1.5
min_sep = 2.5 * RADIUS
clocs = np.zeros((NUM_CYL, 2))
for i in range(NUM_CYL):
    for _ in range(10000):
        x = margin + rng.rand() * (PERIOD - 2*margin)
        y = margin + rng.rand() * (thickness - 2*margin)
        if i == 0 or np.all(np.sqrt((x - clocs[:i, 0])**2 + (y - clocs[:i, 1])**2) > min_sep):
            clocs[i] = [x, y]
            break

sp = smatrix_parameters(WAVELENGTH, PERIOD, PHIINC,
                        1e-11, 1e-4, 5, 3, 1000, 3, 5, 1, PERIOD/120)
cmmaxs = np.full(NUM_CYL, CMMAX, dtype=int)
cepmus = np.column_stack([np.full(NUM_CYL, N_CYL_REF**2), np.full(NUM_CYL, MU)])
crads  = np.full(NUM_CYL, RADIUS)

print(f"Computing S-matrix ({NUM_CYL} dielectric cylinders)...")
t0 = time.time()
S, _ = smatrix(clocs, cmmaxs, cepmus, crads, PERIOD, WAVELENGTH,
               nmax, thickness, sp, 'On')
print(f"Done in {time.time()-t0:.1f}s")

## 2. Transmission Eigenvalue Distribution

In [ ]:
S11 = S[:nm, :nm]
S21 = S[nm:, :nm]

S21T = S21[n_eva:-n_eva, n_eva:-n_eva] if n_eva > 0 else S21
tau = svd(S21T, compute_uv=False)

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(tau, bins=50, range=(0, 1.05), density=True, color='navy', edgecolor='navy')
ax.set_xlabel(r'$\tau$')
ax.set_ylabel(r'$f(\tau)$')
ax.set_title(f'{NUM_CYL} Dielectric Cylinders (n={N_CYL_REF})')
plt.tight_layout()
plt.savefig('dielectric_tau_distribution.png', dpi=150)
plt.show()

## 3. Build Wave Field

We compute the field in three regions using only propagating Floquet modes:
- **Reflection** ($y < 0$): incident + backscattered
- **Slab interior** ($0 \leq y \leq d$): interpolated forward/backward modes
- **Transmission** ($y > d$): transmitted modes

In [ ]:
def build_field(S, nmax, n_eva, thickness, mode, clocs, crads, cmmaxs, cepmus, sp):
    """
    Compute the wave field in three regions:
      - Reflection (y<0):       incident Floquet + reflected Floquet modes
      - Transmission (y>d):     transmitted Floquet modes
      - Interior (0<=y<=d):     incident Floquet + Hankel expansion from each
                                cylinder (0 inside cylinders)
    Re-solves the multiple scattering system to obtain per-cylinder scattering
    coefficients, matching the Julia wave_field_movie reference.
    """
    nm   = 2 * nmax + 1
    S11  = S[:nm, :nm]
    S21  = S[nm:, :nm]
    prop = slice(n_eva, nm - n_eva) if n_eva > 0 else slice(None)

    # ── Input wavefront ──────────────────────────────────────────────────────
    R_tr = S11[n_eva:nm-n_eva, n_eva:nm-n_eva] if n_eva > 0 else S11
    T_tr = S21[n_eva:nm-n_eva, n_eva:nm-n_eva] if n_eva > 0 else S21

    if mode == 'opt_trans':
        _, _, Vh = svd(R_tr)
        v_in = Vh.conj().T[:, -1]
        tc   = np.sum(np.abs(T_tr @ v_in)**2)
        label = f'Optimal Wavefront — {tc*100:.1f}% transmitted'
    else:
        v_in        = np.zeros(R_tr.shape[0], dtype=complex)
        v_in[R_tr.shape[0] // 2] = 1.0
        tc          = np.sum(np.abs(T_tr[:, R_tr.shape[0]//2])**2)
        label = f'Normal Incidence — {tc*100:.1f}% transmitted'

    Input = np.zeros(nm, dtype=complex)
    Input[n_eva:nm-n_eva] = v_in if n_eva > 0 else v_in
    if n_eva == 0:
        Input = v_in

    # ── Floquet mode parameters ───────────────────────────────────────────────
    k    = 2 * np.pi / WAVELENGTH
    m_f  = np.arange(-nmax, nmax + 1)
    kxs  = 2 * np.pi / PERIOD * m_f
    kys  = ky(k, kxs.astype(complex))
    P1   = np.diag(1.0 / np.sqrt(kys / k))
    Inc_c = P1 @ Input
    Ref_c = P1 @ (S11 @ Input)
    Tra_c = P1 @ (S21 @ Input)

    # ── Re-solve multiple scattering → per-cylinder coefficients c_vector ────
    total_steps = len(cmmaxs) * (len(cmmaxs) - 1) // 2 + 2 * nm
    t_mat  = transall(clocs, cmmaxs, PERIOD, WAVELENGTH, sp['phiinc'], sp, total_steps)
    s_vec  = sall(cmmaxs, cepmus, crads, WAVELENGTH)
    z      = np.eye(len(s_vec)) - np.diag(s_vec) @ t_mat

    kxex_all = np.array([
        _extract_scalar(sp['kxs'][sp['MiddleIndex'] + nin])
        for nin in range(-nmax, nmax + 1)
    ])
    V_up       = _vall_batch(clocs, cmmaxs, kxex_all, k, up_down=1)
    v_excite   = s_vec * (V_up @ Input)          # diag(s) @ V @ Input
    c_vector   = np.linalg.solve(z, v_excite)

    # Cylinder start indices in c_vector (supports variable cmmaxs)
    starts = np.concatenate([[0], np.cumsum(2 * cmmaxs + 1)[:-1]])

    # ── Build grid ────────────────────────────────────────────────────────────
    Nx     = GRID_RES
    Ly     = PR_Y * WAVELENGTH
    y_full = np.linspace(-Ly, thickness + Ly, GRID_RES * 2)
    x_phys = np.linspace(0, PERIOD, Nx)
    Field  = np.zeros((Nx, len(y_full)), dtype=complex)

    for jy, y in enumerate(y_full):
        for ix in range(Nx):
            x = x_phys[ix]

            if y < 0:
                # Reflection region
                Field[ix, jy] = (
                    np.sum(Inc_c[prop] * np.exp(1j * (-kxs[prop]*x - kys[prop]*y))) +
                    np.sum(Ref_c[prop] * np.exp(1j * (-kxs[prop]*x + kys[prop]*y)))
                )

            elif y > thickness:
                # Transmission region
                Field[ix, jy] = np.sum(
                    Tra_c[prop] * np.exp(1j * (-kxs[prop]*x - kys[prop]*(y - thickness)))
                )

            else:
                # Interior: incident Floquet + Hankel expansion from cylinders
                total_f = np.sum(
                    Inc_c[prop] * np.exp(1j * (-kxs[prop]*x - kys[prop]*y))
                )
                inside = False
                for i in range(len(crads)):
                    dx = x - clocs[i, 0]
                    dy = y - clocs[i, 1]
                    r  = np.sqrt(dx**2 + dy**2)
                    if r <= crads[i]:
                        inside = True
                        break
                    theta  = np.arctan2(dy, dx)
                    m_max_i = int(cmmaxs[i])
                    i0      = starts[i]
                    for m_idx in range(-m_max_i, m_max_i + 1):
                        total_f += (c_vector[i0 + m_idx + m_max_i] *
                                    hankel2_scipy(m_idx, k * r) *
                                    np.exp(1j * m_idx * theta))
                Field[ix, jy] = 0.0 if inside else total_f

    return Field, x_phys / WAVELENGTH, y_full / WAVELENGTH, label


In [ ]:
print("Building normal incidence field (includes cylinder scattering)...")
t0 = time.time()
F_norm, x_grid, y_grid, label_norm = build_field(
    S, nmax, n_eva, thickness, 'normal', clocs, crads, cmmaxs, cepmus, sp)
print(f"  {label_norm}  [{time.time()-t0:.1f}s]")

print("Building optimal wavefront field...")
t0 = time.time()
F_opt, _, _, label_opt = build_field(
    S, nmax, n_eva, thickness, 'opt_trans', clocs, crads, cmmaxs, cepmus, sp)
print(f"  {label_opt}  [{time.time()-t0:.1f}s]")


## 4. Plot Wave Fields

In [ ]:
vmax = max(np.max(np.abs(np.real(F_norm))), np.max(np.abs(np.real(F_opt)))) * 0.7

fig, axes = plt.subplots(2, 1, figsize=(10, 8))

for ax, F, label in zip(axes, [F_norm, F_opt], [label_norm, label_opt]):
    im = ax.pcolormesh(y_grid, x_grid, np.real(F), cmap='jet', vmin=-vmax, vmax=vmax)
    ax.axvline(0, color='magenta', lw=2)
    ax.axvline(thickness / WAVELENGTH, color='magenta', lw=2)
    ax.set_xlabel('y/lambda')
    ax.set_ylabel('x/lambda')
    ax.set_title(label)
    ax.set_aspect('equal')
    plt.colorbar(im, ax=ax)

plt.tight_layout()
plt.savefig('dielectric_wave_field.png', dpi=150)
plt.show()

## 5. Wave Animation

Animate the wave field by sweeping phase over one full cycle:
`Re(F · exp(-iφ))` for φ ∈ [0, 2π]. Both modes shown simultaneously.

In [ ]:
import matplotlib.animation as animation

N_FRAMES = 40
phases = np.linspace(0, 2*np.pi, N_FRAMES, endpoint=False)

fig, axes = plt.subplots(2, 1, figsize=(10, 8))

vmax_anim = max(np.max(np.abs(F_norm)), np.max(np.abs(F_opt))) * 0.7

ims = []
for ax, F, label in zip(axes, [F_norm, F_opt], [label_norm, label_opt]):
    im = ax.pcolormesh(y_grid, x_grid, np.real(F), cmap='jet',
                       vmin=-vmax_anim, vmax=vmax_anim, shading='auto')
    ax.axvline(0, color='magenta', lw=2)
    ax.axvline(thickness / WAVELENGTH, color='magenta', lw=2)
    # Draw cylinder outlines
    theta = np.linspace(0, 2*np.pi, 40)
    for i in range(NUM_CYL):
        cx = clocs[i, 1] / WAVELENGTH
        cy = clocs[i, 0] / WAVELENGTH
        r_lam = RADIUS / WAVELENGTH
        ax.plot(cx + r_lam * np.cos(theta), cy + r_lam * np.sin(theta), 'k-', lw=0.5)
    ax.set_xlabel(r'y/$\lambda$')
    ax.set_ylabel(r'x/$\lambda$')
    ax.set_title(label)
    ax.set_aspect('equal')
    plt.colorbar(im, ax=ax)
    ims.append(im)

plt.tight_layout()

def update(frame):
    phi = phases[frame]
    for im, F in zip(ims, [F_norm, F_opt]):
        im.set_array(np.real(F * np.exp(+1j * phi)))   # +1j: wave travels in +y
    return ims

anim = animation.FuncAnimation(fig, update, frames=N_FRAMES, interval=80, blit=True)

print("Saving dielectric_wave_animation.mp4 ...")
anim.save('dielectric_wave_animation.mp4', writer='ffmpeg', fps=15, dpi=120)
print("Saved: dielectric_wave_animation.mp4")
plt.close(fig)
